# Ingestão de Dados: Batch vs Streaming

In [0]:
%sql
-- Remove o schema
DROP SCHEMA IF EXISTS senai.batch CASCADE;

In [0]:
%sql
-- Configura catálogo, schema e volume para armazenar arquivos
CREATE CATALOG IF NOT EXISTS senai;
CREATE SCHEMA IF NOT EXISTS senai.batch;
CREATE VOLUME IF NOT EXISTS senai.batch.volume;

In [0]:
# Primeira carga: exporta dados de vendas para arquivo CSV
import pandas as pd

df = (
    spark.table("samples.bakehouse.sales_transactions")
         .filter("DATE(dateTime) BETWEEN '2024-05-01' AND '2024-05-14'")
)

pdf = df.toPandas()
pdf.to_csv("/Volumes/senai/batch/volume/sales1.csv", index=False)

print(f'Registros: {df.count()}')

## CTAS (Create Table As Select)

- Substitui todos os dados a cada execução
- **Não incremental**: utilizado para processar todos os registros em cada execução

In [0]:
%sql
-- Cria tabela, carregando todos os arquivos CSV do volume
-- TODO

In [0]:
# Segunda carga: adiciona novos dados
df = (
    spark.table("samples.bakehouse.sales_transactions")
         .filter("DATE(dateTime) BETWEEN '2024-05-15' AND '2024-05-31'")
)

pdf = df.toPandas()
pdf.to_csv("/Volumes/senai/batch/volume/sales2.csv", index=False)

print(f'Registros: {df.count()}')

In [0]:
%sql
-- Recria a tabela incluindo todos os arquivos novamente, incluindo os dados já carregados
-- TODO

# ATIVIDADE 1

Execute uma etapa de ingestão incremental com PySpark, utilizando `.mode('append')`, na tabela anterior `senai.batch.ctas`

1. Configure as opções `header` e `inferSchema` como verdadeiras
2. Utilize `.csv('/Volumes/senai/batch/volume/')`
3. Conte a quantidade final de registros com `spark.table('senai.senai.ctas').count()`

In [0]:
# Atividade 1


## COPY INTO

- **COPY INTO**: suporta **idempotência** e cargas incrementais

In [0]:
%sql
-- Recria schema, volume e tabela vazia
DROP SCHEMA IF EXISTS senai.batch CASCADE;
CREATE SCHEMA IF NOT EXISTS senai.batch;
CREATE VOLUME IF NOT EXISTS senai.batch.volume;
CREATE TABLE IF NOT EXISTS senai.batch.copy_into;

In [0]:
# Primeira carga: exporta dados de vendas para arquivo CSV
df = (
    spark.table("samples.bakehouse.sales_transactions")
         .filter("DATE(dateTime) BETWEEN '2024-05-01' AND '2024-05-14'")
)

pdf = df.toPandas()
pdf.to_csv("/Volumes/senai/batch/volume/sales1.csv", index=False)

print(f'Registros: {df.count()}')

In [0]:
%sql
-- Primeira carga incremental
-- TODO

In [0]:
# Segunda carga: adiciona novos dados
df = (
    spark.table("samples.bakehouse.sales_transactions")
         .filter("DATE(dateTime) BETWEEN '2024-05-15' AND '2024-05-31'")
)

pdf = df.toPandas()
pdf.to_csv("/Volumes/senai/batch/volume/sales2.csv", index=False)  # Novo arquivo

print(f'Registros: {df.count()}')

In [0]:
%sql
-- Segunda execução (carga incremental), processando apenas os arquivos novos
-- TODO

In [0]:
# Verifica quantidade final de registros após as cargas incrementais
# TODO

### ATIVIDADE 2

Simule carga usando **COPY INTO** com a tabela `samples.bakehouse.media_customer_reviews`

1. Crie tabela vazia `workspace.default.reviews_copy_into`
2. Carrege o coneúdo da tabela de origem para um arquivo específico
3. Leia o arquivo e carregue as reviews utilizando `COPY INTO`
4. Verifique a quantidade de reviews carregadas

In [0]:
# Atividade 2


# Auto Loader

- Ingestão **incremental e automatizada** de arquivos
- Detecta **novos arquivos automaticamente** em cloud storage
- Suporta **evolução de esquema** (schema evolution)
- Ideal para **streaming**

In [0]:
%sql
-- Recria estrutura limpa para streaming incremental
DROP SCHEMA IF EXISTS senai.batch CASCADE;
CREATE SCHEMA IF NOT EXISTS senai.batch;
CREATE VOLUME IF NOT EXISTS senai.batch.volume;
CREATE TABLE IF NOT EXISTS senai.batch.auto_loader;  -- Tabela de destino

In [0]:
# Prepara arquivos para demonstração
# Arquivo 1: primeira quinzena de maio
df = (
    spark.table("samples.bakehouse.sales_transactions")
         .filter("DATE(dateTime) BETWEEN '2024-05-01' AND '2024-05-14'")
)

pdf = df.toPandas()
pdf.to_csv("/Volumes/senai/batch/volume/sales1.csv", index=False)

# Arquivo 2: segunda quinzena de maio
df = (
    spark.table("samples.bakehouse.sales_transactions")
         .filter("DATE(dateTime) BETWEEN '2024-05-15' AND '2024-05-31'")
)

pdf = df.toPandas()
pdf.to_csv("/Volumes/senai/batch/volume/sales2.csv", index=False)

In [0]:
# Auto Loader: Ingestão incremental automatizada com streaming
# TODO

### Resumo Comparativo

| Característica | CTAS | COPY INTO | Auto Loader |
|----------------|------|-----------|-------------|
| **Comportamento** | ❌ **Substitui** | ✅ **Adiciona** | ✅ **Incremental** |
| **Execução** | Manual/Agendada | Manual/Agendada | Automática contínua |
| **Idempotência** | ❌ Não | ✅ Sim (rastreia arquivos) | ✅ Sim (rastreia arquivos) |
| **Streaming** | ❌ Não | ❌ Não | ✅ Sim |

---

### 🎯 Quando usar cada um?

* **CTAS**: 
  * Sistemas legados
  * Criar snapshots de dados
  
* **COPY INTO**: 
  * Carregar dados históricos de arquivos
  * ETL batch incremental agendado
  * Migração de dados de sistemas externos
  
* **Auto Loader**: 
  * Pipelines de dados contínuos
  * Ingestão lakehouse em tempo real
  * Quando novos arquivos chegam constantemente
  * Precisa de detecção automática de novos dados